In [2]:
# Loading 
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import symbols, diff, solve, lambdify
from sympy.physics.quantum.tensorproduct import TensorProduct
# Define the variable and the function
ri = symbols('ri')
phi = symbols('phi')
lbd = symbols('lbd')
R = symbols('R')
p = symbols('p')


In [16]:
# Initialize the parameters
L = 0.05
R_i = 0.00035
R_o = 0.0023
Theta = 0
alpha = 1.44
mu_1 = 427560
mu_2 = 8500000
P_array = np.linspace(0, 2000000, 100)  # Pressure range from 0 to 2000 kPa

In [17]:
# Define the function f(ri, phi, R, lambda) using sympy
# Deformation gradient tensor
r = sp.sqrt((R**2-R_i**2)/lbd + ri**2)
F = sp.Matrix([[R/(r*lbd), 0, 0], [0, r/R, r*phi/L], [0, 0, lbd]])
S = sp.Matrix([[0], [sp.sin(alpha)], [sp.cos(alpha)]])
print("Deformation Gradient Tensor F:")
sp.pprint(F)
print("\nDirection Vector s:")
sp.pprint(S)
# Calculate the Cauchy-Green deformation tensor C
s = F * S
sp.pprint(s)

Deformation Gradient Tensor F:
⎡              R                                                               ↪
⎢──────────────────────────────              0                               0 ↪
⎢         _____________________                                                ↪
⎢        ╱        2                                                            ↪
⎢       ╱    2   R  - 1.225e-7                                                 ↪
⎢lbd⋅  ╱   ri  + ─────────────                                                 ↪
⎢    ╲╱               lbd                                                      ↪
⎢                                                                              ↪
⎢                                     _____________________                    ↪
⎢                                    ╱        2                                ↪
⎢                                   ╱    2   R  - 1.225e-7               _____ ↪
⎢                                  ╱   ri  + ─────────────              ╱     

In [18]:
# First and fourth invariants of C
I1 = sp.trace(sp.MatMul(F, F.T))
I4 = sp.MatMul(s.T, s)
print(sp.shape(I4))   
I4 = I4[0,0]
sp.pprint(I4)

(1, 1)
                                                                               ↪
                          ⎛                                                    ↪
                          ⎜                                                    ↪
                          ⎜                        _____________________       ↪
                          ⎜                       ╱        2               0.9 ↪
                      2   ⎜                      ╱    2   R  - 1.225e-7        ↪
0.0170103438010126⋅lbd  + ⎜2.60847417476291⋅φ⋅  ╱   ri  + ─────────────  + ─── ↪
                          ⎝                   ╲╱               lbd             ↪

↪                                           2
↪                     _____________________⎞ 
↪                    ╱        2            ⎟ 
↪                   ╱    2   R  - 1.225e-7 ⎟ 
↪ 91458348191686⋅  ╱   ri  + ───────────── ⎟ 
↪                ╲╱               lbd      ⎟ 
↪ ─────────────────────────────────────────⎟ 
↪                   R 

In [19]:
sigma_tube = mu_1 * sp.MatMul(F, F.T) 
sigma_coil = mu_2 * TensorProduct(s, s.T) * (sp.sqrt(I4) - 1) / sp.sqrt(I4) 
sigma_total = sigma_tube + sigma_coil - (p * sp.eye(3))
sp.shape(sigma_total)

(3, 3)

In [20]:
f_pressure = R / (R**2 - R_i**2 + lbd*ri**2) * (sigma_total[1,1] - sigma_total[0,0])
df_pressure_dri = diff(f_pressure, ri)
df_pressure_dphi = diff(f_pressure, phi)
df_pressure_dlbd = diff(f_pressure, lbd)
f_load = np.pi * R/lbd * (2 * sigma_total[2,2] - sigma_total[0,0] - sigma_total[1,1])
df_load_dri = diff(f_load, ri)
df_load_phi = diff(f_load, phi)
df_load_dlbd = diff(f_load, lbd)
f_moment = sigma_total[1,2] * r * R / lbd * 2 * np.pi
df_moment_dri = diff(f_moment, ri)
df_moment_dphi = diff(f_moment, phi)
df_moment_dlbd = diff(f_moment, lbd)

In [57]:
# Initialize values for Newton-Raphson method
lbd_init = 1.5
phi_init = 0.5
ri_init = 0.0005
subs_init = {lbd: lbd_init, phi: phi_init, ri: ri_init}
tolerance = 1e-6
max_iterations = 100


In [11]:
# Numerical evaluation of the integrals
def numerical_integral(func, x_start, x_end, num_points=100):
    x_values = np.linspace(x_start, x_end, num_points)
    y_values = np.array([func.subs(R, x).evalf() for x in x_values], dtype=np.float64)
    integral = np.sum(y_values) * (x_end - x_start) / num_points
    return integral


In [ ]:
# analytical integral for pressure and load and moment
def Jacobian_matrix(f_pressure, f_load, f_moment, df_pressure_dri, df_pressure_dphi, df_pressure_dlbd,
                    df_load_dri, df_load_phi, df_load_dlbd, df_moment_dri, df_moment_dphi, df_moment_dlbd,
                    ri_val, phi_val, lbd_val, R_i, R_o, P):

    subs_val = [(ri, ri_val), (phi, phi_val), (lbd, lbd_val)]
    f_pressure_subs = f_pressure.subs(subs_val)
    f_load_subs = f_load.subs(subs_val)
    f_moment_subs = f_moment.subs(subs_val)
    df_pressure_dri_subs = df_pressure_dri.subs(subs_val)
    df_pressure_dphi_subs = df_pressure_dphi.subs(subs_val)
    df_pressure_dlbd_subs = df_pressure_dlbd.subs(subs_val)
    df_load_dri_subs  = df_load_dri.subs(subs_val)
    df_load_dphi_subs = df_load_phi.subs(subs_val)
    df_load_dlbd_subs = df_load_dlbd.subs(subs_val)
    df_moment_dri_subs = df_moment_dri.subs(subs_val)
    df_moment_dphi_subs = df_moment_dphi.subs(subs_val)
    df_moment_dlbd_subs = df_moment_dlbd.subs(subs_val)
    # seperate into thread into threads to speed up the computation of the integrals
    
    f_pressure_integral = numerical_integral(f_pressure_subs, R_i, R_o) - P
    f_load_integral = numerical_integral(f_load_subs, R_i, R_o)
    f_moment_integral = numerical_integral(f_moment_subs, R_i, R_o)
    df_pressure_dri_integral = numerical_integral(df_pressure_dri_subs, R_i, R_o)
    df_pressure_dphi_integral = numerical_integral(df_pressure_dphi_subs, R_i, R_o)
    df_pressure_dlbd_integral = numerical_integral(df_pressure_dlbd_subs, R_i, R_o)
    df_load_dri_integral = numerical_integral(df_load_dri_subs, R_i, R_o)
    df_load_dphi_integral = numerical_integral(df_load_dphi_subs, R_i, R_o)
    df_load_dlbd_integral = numerical_integral(df_load_dlbd_subs, R_i, R_o)
    df_moment_dri_integral = numerical_integral(df_moment_dri_subs, R_i, R_o)
    df_moment_dphi_integral = numerical_integral(df_moment_dphi_subs, R_i, R_o)
    df_moment_dlbd_integral = numerical_integral(df_moment_dlbd_subs, R_i, R_o)         

    J = np.array([[df_pressure_dri_integral, df_pressure_dphi_integral, df_pressure_dlbd_integral],
                [df_load_dri_integral, df_load_dphi_integral, df_load_dlbd_integral],
                [df_moment_dri_integral, df_moment_dphi_integral, df_moment_dlbd_integral]])
    
    return J, f_pressure_integral, f_load_integral, f_moment_integral

# Newton-Raphson iteration
def newton_raphson(f_pressure, f_load, f_moment, df_pressure_dri, df_pressure_dphi, df_pressure_dlbd,
                    df_load_dri, df_load_phi, df_load_dlbd, df_moment_dri, df_moment_dphi, df_moment_dlbd,
                    ri_val, phi_val, lbd_val, R_i, R_o, P, lr=0.05, max_iterations=100, tolerance=0.0001):

    x = np.array([ri_init, phi_init, lbd_init], dtype=np.float64)
    f_pressure_history = []
    f_load_history = []
    f_moment_history = []
    for i in range(max_iterations):
        ri_val, phi_val, lbd_val = x
        J, f_pressure_integral, f_load_integral, f_moment_integral = Jacobian_matrix(f_pressure, f_load, f_moment, df_pressure_dri, df_pressure_dphi, df_pressure_dlbd,
                                                                            df_load_dri, df_load_phi, df_load_dlbd, df_moment_dri, df_moment_dphi, df_moment_dlbd,
                                                                            ri_val, phi_val, lbd_val, R_i, R_o, P)
        F = np.array([f_pressure_integral, f_load_integral, f_moment_integral], dtype=np.float64)
        f_pressure_history.append(f_pressure_integral)
        f_load_history.append(f_load_integral)
        f_moment_history.append(f_moment_integral)
        delta = -np.dot(np.linalg.inv(J), F)
        x += lr * delta
        # Display the current iteration and the values of ri, phi, lambda
        print(f"Iteration {i+1}: ri = {x[0]}, phi = {x[1]}, lambda = {x[2]}")
        if np.linalg.norm(delta) < tolerance:
            print(f"Convergence achieved after {i+1} iterations.")
            break
    return x, f_pressure_history, f_load_history, f_moment_history

In [ ]:
# Loop through all pressure value from 50kPa to 1MPa
P_array = np.linspace(50000, 1000000, 20)
ri_array = np.zeros_like(P_array)
phi_array = np.zeros_like(P_array)
lbd_array = np.zeros_like(P_array)

for idx in range(len(P_array)):
    P = P_array[i]
    solution, f_pressure_history, f_load_history, f_moment_history = newton_raphson(f_pressure, f_load, f_moment, df_pressure_dri, df_pressure_dphi, df_pressure_dlbd,
                                                                            df_load_dri, df_load_phi, df_load_dlbd, df_moment_dri, df_moment_dphi, df_moment_dlbd,
                                                                            ri_init, phi_init, lbd_init, R_i, R_o, P, lr=0.05, max_iterations=200, tolerance=1e-6)
    ri_solution, phi_solution, lbd_solution = solution
    print("=====================================================================================================================")
    print(f"Pressure: P = {P:10d} Solution: ri = {ri_solution:10.5f}, phi = {phi_solution:10.5f}, lambda = {lbd_solution:10.5f}")
    print("=====================================================================================================================")
    ri_array[idx] = ri_solution
    phi_array[idx] = phi_solution
    lbd_array[idx] = lbd_solution


SyntaxError: invalid syntax (3351321093.py, line 4)

In [ ]:
# Plot differences of tube configuration respect to input pressure
plt.subplot(1,3,1)
plt.plot(P_array, ri_array, marker="o")
plt.title("Radius vs Pressure")
plt.xlabel("Pressure (Pa)")
plt.ylabel("Radius of tube (mm)")
plt.subplot(1,3,2)
plt.plot(P_array, phi_array, marker="x")
plt.title("Twist angle vs Pressure")
plt.xlabel("Pressure (Pa)")
plt.ylabel("Twist angle of tube (rad)")
plt.subplot(1,3,3)
plt.plot(P_array, lbd_array, marker="+")
plt.title("Elongation vs Pressure")
plt.xlabel("Pressure (Pa)")
plt.ylabel("Elongation of tube")
plt.legend()
plt.grid()
plt.show()
plt.plot()
